# 16. RF Exploration

Notebook per esplorare il dataset RF e avviare una pipeline riproducibile di analisi preliminare.

## 1. Configurazione ambiente e dipendenze

Import delle librerie necessarie e verifica veloce dell'ambiente Python.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Python: {sys.version.split()[0]}")
print(f"Pandas: {pd.__version__}")
print(f"Numpy: {np.__version__}")

sns.set_theme(style="whitegrid")

## 2. Definizione costanti e configurazione runtime

Definizione dei percorsi e delle tabelle target per il confronto clinica-volumi.

In [ ]:
PROJECT_ROOT = Path("/home/mario/Repository/Normal_Alzeihmer")
RF_DIR = PROJECT_ROOT / "data" / "RF"
VOLUMES_DIR = PROJECT_ROOT / "data" / "volumes"

CLINICAL_FILE = RF_DIR / "clinical_informations.csv"
RF_VOLUME_FILES = {
    "rf_real": VOLUMES_DIR / "rf_real_all_volumes.csv",
    "rf_generated": VOLUMES_DIR / "rf_generated_all_volumes.csv",
    "rf_ct": VOLUMES_DIR / "rf_ct_all_volumes.csv",
}

print(f"Clinical file exists: {CLINICAL_FILE.exists()} ({CLINICAL_FILE})")
for name, path in RF_VOLUME_FILES.items():
    print(f"{name:12s} exists: {path.exists()} ({path.name})")

## 3. Implementazione del nucleo applicativo

Funzioni riusabili per normalizzare colonne, trovare subject ID e preparare report.

In [ ]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize column names for robust downstream access."""
    out = df.copy()
    out.columns = [c.strip().lower().replace(" ", "_") for c in out.columns]
    return out


def find_subject_column(df: pd.DataFrame) -> str | None:
    """Best-effort subject id column detection."""
    candidates = [
        "subject_id",
        "subject",
        "subjectid",
        "subject_id_x",
        "subject_id_y",
        "subjectid",
    ]
    for col in df.columns:
        if col in candidates:
            return col
    for col in df.columns:
        if "subject" in col:
            return col
    return None


def load_clinical_table(path: Path) -> pd.DataFrame:
    """Load RF clinical table with normalized names."""
    if not path.exists():
        raise FileNotFoundError(f"Clinical file not found: {path}")
    raw = pd.read_csv(path)
    clinical = normalize_columns(raw)

    if "subject_id" not in clinical.columns and "subject_id" in [c.replace(" ", "_") for c in raw.columns]:
        pass

    return clinical


def load_available_volume_tables(file_map: dict[str, Path]) -> dict[str, pd.DataFrame]:
    """Load only the RF volume tables that exist on disk."""
    loaded: dict[str, pd.DataFrame] = {}
    for name, path in file_map.items():
        if not path.exists():
            continue
        loaded[name] = normalize_columns(pd.read_csv(path))
    return loaded


def overlap_report(clinical_df: pd.DataFrame, volume_df: pd.DataFrame) -> dict[str, int | float]:
    """Compute subject overlap metrics between clinical and volume table."""
    clinical_sub_col = find_subject_column(clinical_df)
    volume_sub_col = find_subject_column(volume_df)

    if clinical_sub_col is None or volume_sub_col is None:
        return {
            "clinical_n": len(clinical_df),
            "volume_n": len(volume_df),
            "clinical_unique": 0,
            "volume_unique": 0,
            "overlap": 0,
            "overlap_pct_on_clinical": 0.0,
        }

    clinical_ids = set(clinical_df[clinical_sub_col].astype(str))
    volume_ids = set(volume_df[volume_sub_col].astype(str))
    overlap = clinical_ids.intersection(volume_ids)

    return {
        "clinical_n": len(clinical_df),
        "volume_n": len(volume_df),
        "clinical_unique": len(clinical_ids),
        "volume_unique": len(volume_ids),
        "overlap": len(overlap),
        "overlap_pct_on_clinical": round(100.0 * len(overlap) / max(len(clinical_ids), 1), 2),
    }

## 4. Gestione input/output e validazione

Caricamento tabelle, controlli di validita` e preparazione dei riepiloghi.

In [ ]:
clinical_df = load_clinical_table(CLINICAL_FILE)
volume_tables = load_available_volume_tables(RF_VOLUME_FILES)

print("Clinical shape:", clinical_df.shape)
print("Clinical columns:", list(clinical_df.columns))
print("\nMissing values per colonna clinica:")
print(clinical_df.isna().sum().to_string())

required_cols = ["subject_id", "sex", "age", "diagnosi"]
missing_required = [c for c in required_cols if c not in clinical_df.columns]
print("\nRequired columns mancanti:", missing_required if missing_required else "nessuna")

print("\nTabelle volumi RF caricate:")
if volume_tables:
    for name, df in volume_tables.items():
        print(f"- {name:12s}: {df.shape}")
else:
    print("Nessuna tabella volumi RF trovata.")

## 5. Esecuzione del flusso principale

Riepilogo demografico, grafici base e confronto overlap tra clinica e volumi.

In [ ]:
# Demografia clinica
if "age" in clinical_df.columns:
    print("Age summary:")
    print(clinical_df["age"].describe().to_string())
    print()

if "sex" in clinical_df.columns:
    print("Sex distribution:")
    print(clinical_df["sex"].value_counts(dropna=False).to_string())
    print()

if "diagnosi" in clinical_df.columns:
    print("Diagnosis distribution:")
    print(clinical_df["diagnosi"].value_counts(dropna=False).to_string())

# Grafici principali
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if "age" in clinical_df.columns:
    axes[0].hist(clinical_df["age"].dropna(), bins=20, color="#2a9d8f", edgecolor="white")
    axes[0].set_title("Distribuzione Eta`")
    axes[0].set_xlabel("Eta`")
    axes[0].set_ylabel("Conteggio")
else:
    axes[0].text(0.5, 0.5, "Colonna age non disponibile", ha="center", va="center")
    axes[0].set_axis_off()

if "diagnosi" in clinical_df.columns:
    diag_counts = clinical_df["diagnosi"].value_counts()
    axes[1].bar(diag_counts.index.astype(str), diag_counts.values, color="#264653")
    axes[1].set_title("Distribuzione Diagnosi")
    axes[1].set_xlabel("Diagnosi")
    axes[1].set_ylabel("Conteggio")
    axes[1].tick_params(axis="x", rotation=45)
else:
    axes[1].text(0.5, 0.5, "Colonna diagnosi non disponibile", ha="center", va="center")
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()

# Overlap clinica-volumi
overlap_rows = []
for name, df in volume_tables.items():
    row = overlap_report(clinical_df, df)
    row["table"] = name
    overlap_rows.append(row)

overlap_df = pd.DataFrame(overlap_rows)
if not overlap_df.empty:
    cols = ["table", "clinical_n", "volume_n", "clinical_unique", "volume_unique", "overlap", "overlap_pct_on_clinical"]
    overlap_df = overlap_df[cols].sort_values("table")
    print("\nOverlap clinica-volumi:")
    print(overlap_df.to_string(index=False))
else:
    print("\nNessun confronto overlap disponibile (volumi RF assenti).")

## 6. Test rapidi e debug in notebook

Assert minimi per verificare coerenza delle tabelle e facilitare il troubleshooting.

In [ ]:
# Test minimi
assert len(clinical_df) > 0, "Clinical table vuota"

for col in ["subject_id", "sex", "age", "diagnosi"]:
    if col not in clinical_df.columns:
        print(f"[WARN] Colonna clinica non trovata: {col}")

if "age" in clinical_df.columns:
    assert clinical_df["age"].dropna().ge(0).all(), "Trovata eta` negativa"

for name, df in volume_tables.items():
    sub_col = find_subject_column(df)
    if sub_col is None:
        print(f"[WARN] Nessuna colonna soggetto identificata in {name}")
    else:
        dup_n = int(df[sub_col].astype(str).duplicated().sum())
        print(f"[INFO] {name}: duplicati subject={dup_n}")

print("\nQuick checks completati.")

## Next Steps

1. Se necessario, uniformare la chiave soggetto tra RF clinico e tabelle volumi (es. trim, cast string).
2. Salvare il report overlap in CSV dentro `data/RF/evaluations/` per audit ripetibile.
3. Integrare i risultati RF nei notebook di concordanza multimodale, mantenendo RF separato dal training normativo.
4. Se in futuro RF entra nel pipeline normativo, aggiungere un branch esplicito con policy QC dedicata.